# Setup: Create Warehouse and Tables

Run this notebook **once** after `docker-compose up` to create the demo data.

After this, open `demo.ipynb` or `governance.ipynb`.

In [1]:
import os, socket

def _in_docker():
    try:
        socket.getaddrinfo("lakekeeper", 8181)
        return True
    except socket.gaierror:
        return False

IN_DOCKER = _in_docker()
LK_HOST = "lakekeeper" if IN_DOCKER else "localhost"
PROXY_HOST = "labels-proxy" if IN_DOCKER else "localhost"
MINIO_INTERNAL = "http://minio:9000"

LK_URL = f"http://{LK_HOST}:8181"
CATALOG_URL = f"http://{LK_HOST}:8181/catalog"
PROXY_URL = f"http://{PROXY_HOST}:8182"

print(f"Environment:  {'Docker' if IN_DOCKER else 'Local'}")
print(f"Lakekeeper:   {LK_URL}")
print(f"Labels Proxy: {PROXY_URL}")


Environment:  Local
Lakekeeper:   http://localhost:8181
Labels Proxy: http://localhost:8182


## Step 1: Create Warehouse

In [ ]:
import httpx

client = httpx.Client(base_url=LK_URL, timeout=30)

# Step 1: Ensure project exists
resp = client.post("/management/v1/project", json={"project-name": "default"})
if resp.status_code in (200, 201):
    project_id = resp.json()["project-id"]
    print(f"Created project (id={project_id})")
elif resp.status_code == 409:
    # Already exists — but we need the ID. Lakekeeper doesn't have a list projects API,
    # so we store it from the first run. Try creating warehouse to find out.
    print("Project already exists")
    project_id = None
else:
    print(f"Project creation response: {resp.status_code} {resp.text}")
    project_id = None

# Step 2: Check if warehouse exists
resp = client.get("/management/v1/warehouse")
warehouses = resp.json().get("warehouses", []) if resp.status_code == 200 else []
existing = [w for w in warehouses if (w.get("warehouse-name") or w.get("name")) == "healthcare"]

if existing:
    print(f"Warehouse 'healthcare' already exists (id={existing[0]['id']})")
else:
    if project_id is None:
        # Get project-id from any existing warehouse, or use the error to find it
        if warehouses:
            project_id = warehouses[0].get("project-id")
        else:
            # No warehouses and no project_id — the project creation returned 409
            # but we need the id. Re-create with a unique name to get any project id.
            print("ERROR: Cannot determine project-id. Please restart with: docker-compose down -v && docker-compose up -d")
            raise SystemExit(1)

    resp = client.post("/management/v1/warehouse", json={
        "warehouse-name": "healthcare",
        "project-id": project_id,
        "storage-profile": {
            "type": "s3",
            "bucket": "warehouse",
            "region": "us-east-1",
            "endpoint": MINIO_INTERNAL,
            "path-style-access": True,
            "flavor": "minio",
            "sts-enabled": False,
        },
        "storage-credential": {
            "type": "s3",
            "credential-type": "access-key",
            "aws-access-key-id": "admin",
            "aws-secret-access-key": "password",
        },
    })
    if resp.status_code in (200, 201):
        print(f"Created warehouse 'healthcare' (id={resp.json()['id']})")
    else:
        print(f"Warehouse creation failed: {resp.status_code} {resp.text}")

## Step 2: Create Namespace and Tables

In [6]:
from pyiceberg.catalog import load_catalog
from pyiceberg.schema import Schema
from pyiceberg.types import IntegerType, StringType, DateType, DoubleType, NestedField
import pyarrow as pa
from datetime import date

# Connect through proxy (adds labels) to Lakekeeper (stores data)
catalog = load_catalog("healthcare", **{
    "type": "rest",
    "uri": CATALOG_URL,
    "warehouse": "healthcare",
})

try:
    catalog.create_namespace("healthcare")
    print("Created namespace: healthcare")
except Exception as e:
    print(f"Namespace: {e}")


RESTError: WarehouseNotFound: A warehouse 'healthcare' does not exist

In [7]:
# --- Create patients ---
try:
    t = catalog.create_table("healthcare.patients", schema=Schema(
        NestedField(1, "patient_id", LongType(), required=False),
        NestedField(2, "name", StringType(), required=False),
        NestedField(3, "email", StringType(), required=False),
        NestedField(4, "diagnosis", StringType(), required=False),
        NestedField(5, "dob", DateType(), required=False),
        NestedField(6, "insurance_id", StringType(), required=False),
    ))
    t.append(pa.table({
        "patient_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
        "name": ["Alice Johnson", "Bob Smith", "Carol Davis", "Dan Wilson",
                 "Eva Martinez", "Frank Brown", "Grace Lee", "Henry Taylor"],
        "email": ["alice.j@example.com", "bob.s@example.com", "carol.d@example.com",
                  "dan.w@example.com", "eva.m@example.com", "frank.b@example.com",
                  "grace.l@example.com", "henry.t@example.com"],
        "diagnosis": ["Hypertension", "Type 2 Diabetes", "Asthma", "Migraine",
                      "Hypertension", "Anxiety Disorder", "Type 1 Diabetes", "COPD"],
        "dob": [date(1985,3,15), date(1972,7,22), date(1990,11,8), date(1968,1,30),
                date(1995,5,12), date(1980,9,3), date(1988,12,19), date(1955,4,7)],
        "insurance_id": ["INS-AA1001", "INS-BB2002", "INS-CC3003", "INS-DD4004",
                         "INS-EE5005", "INS-FF6006", "INS-GG7007", "INS-HH8008"],
    }))
    print("Created: healthcare.patients (8 rows)")
except Exception as e:
    print(f"patients: {e}")

# --- Create visits_summary ---
try:
    t = catalog.create_table("healthcare.visits_summary", schema=Schema(
        NestedField(1, "visit_date", DateType(), required=False),
        NestedField(2, "department", StringType(), required=False),
        NestedField(3, "visit_count", LongType(), required=False),
        NestedField(4, "avg_wait_time", DoubleType(), required=False),
        NestedField(5, "satisfaction_score", DoubleType(), required=False),
    ))
    t.append(pa.table({
        "visit_date": [date(2026,1,5)]*3 + [date(2026,1,12)]*3 + [date(2026,2,1)]*3 +
                      [date(2026,2,15)]*3 + [date(2026,3,1)]*3,
        "department": ["Cardiology", "Neurology", "General"] * 5,
        "visit_count": [45, 32, 128, 52, 28, 135, 48, 35, 142, 55, 30, 138, 60, 33, 145],
        "avg_wait_time": [22.5, 18.3, 35.2, 24.1, 17.8, 38.5, 21.0, 19.2, 33.8, 25.3, 16.5, 36.1, 23.7, 18.9, 34.5],
        "satisfaction_score": [4.2, 4.5, 3.8, 4.1, 4.6, 3.7, 4.3, 4.4, 3.9, 4.0, 4.7, 3.6, 4.4, 4.5, 3.8],
    }))
    print("Created: healthcare.visits_summary (15 rows)")
except Exception as e:
    print(f"visits_summary: {e}")

# --- Create billing ---
try:
    t = catalog.create_table("healthcare.billing", schema=Schema(
        NestedField(1, "invoice_id", LongType(), required=False),
        NestedField(2, "patient_id", LongType(), required=False),
        NestedField(3, "amount", DoubleType(), required=False),
        NestedField(4, "insurance_code", StringType(), required=False),
        NestedField(5, "billing_date", DateType(), required=False),
    ))
    t.append(pa.table({
        "invoice_id": [5001, 5002, 5003, 5004, 5005, 5006, 5007, 5008, 5009, 5010],
        "patient_id": [1001, 1002, 1003, 1001, 1004, 1005, 1006, 1007, 1008, 1002],
        "amount": [1250.0, 890.5, 425.0, 2100.0, 675.0, 1890.0, 320.0, 1450.0, 3200.0, 780.0],
        "insurance_code": ["BCBS-100", "AETNA-200", "BCBS-100", "BCBS-100", "UHC-300",
                           "CIGNA-400", "AETNA-200", "UHC-300", "BCBS-100", "AETNA-200"],
        "billing_date": [date(2026,1,10), date(2026,1,12), date(2026,1,15), date(2026,1,20),
                         date(2026,1,25), date(2026,2,1), date(2026,2,5), date(2026,2,10),
                         date(2026,2,15), date(2026,2,20)],
    }))
    print("Created: healthcare.billing (10 rows)")
except Exception as e:
    print(f"billing: {e}")


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃    ┃ Table field                      ┃ Dataframe field                  ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ❌ │ 1: patient_id: required int      │ 1: patient_id: optional long     │
│ ❌ │ 2: name: required string         │ 2: name: optional string         │
│ ❌ │ 3: email: required string        │ 3: email: optional string        │
│ ✅ │ 4: diagnosis: optional string    │ 4: diagnosis: optional string    │
│ ❌ │ 5: dob: required date            │ 5: dob: optional date            │
│ ✅ │ 6: insurance_id: optional string │ 6: insurance_id: optional string │
└────┴──────────────────────────────────┴──────────────────────────────────┘

patients: Mismatch in fields:
┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃    ┃ Table field                      ┃ Dataframe field                  ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ❌ │ 1: patient_id: required int      │ 1: patient_id: optional long     │
│ ❌ │ 2: name: required string         │ 2: name: optional string         │
│ ❌ │ 3: email: required string        │ 3: email: optional string        │
│ ✅ │ 4: diagnosis: optional string    │ 4: diagnosis: optional string    │
│ ❌ │ 5: dob: required date            │ 5: dob: optional date            │
│ ✅ │ 6: insurance_id: optional string │ 6: insurance_id: optional string │
└────┴──────────────────────────────────┴──────────────────────────────────┘



┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃    ┃ Table field                            ┃ Dataframe field                        ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ❌ │ 1: visit_date: required date           │ 1: visit_date: optional date           │
│ ❌ │ 2: department: required string         │ 2: department: optional string         │
│ ❌ │ 3: visit_count: required int           │ 3: visit_count: optional long          │
│ ❌ │ 4: avg_wait_time: required double      │ 4: avg_wait_time: optional double      │
│ ❌ │ 5: satisfaction_score: required double │ 5: satisfaction_score: optional double │
└────┴────────────────────────────────────────┴────────────────────────────────────────┘

visits_summary: Mismatch in fields:
┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃    ┃ Table field                            ┃ Dataframe field                        ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ❌ │ 1: visit_date: required date           │ 1: visit_date: optional date           │
│ ❌ │ 2: department: required string         │ 2: department: optional string         │
│ ❌ │ 3: visit_count: required int           │ 3: visit_count: optional long          │
│ ❌ │ 4: avg_wait_time: required double      │ 4: avg_wait_time: optional double      │
│ ❌ │ 5: satisfaction_score: required double │ 5: satisfaction_score: optional double │
└────┴────────────────────────────────────────┴────────────────────────────────────────┘



┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃    ┃ Table field                        ┃ Dataframe field                    ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ❌ │ 1: invoice_id: required int        │ 1: invoice_id: optional long       │
│ ❌ │ 2: patient_id: required int        │ 2: patient_id: optional long       │
│ ❌ │ 3: amount: required double         │ 3: amount: optional double         │
│ ✅ │ 4: insurance_code: optional string │ 4: insurance_code: optional string │
│ ❌ │ 5: billing_date: required date     │ 5: billing_date: optional date     │
└────┴────────────────────────────────────┴────────────────────────────────────┘

billing: Mismatch in fields:
┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃    ┃ Table field                        ┃ Dataframe field                    ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ❌ │ 1: invoice_id: required int        │ 1: invoice_id: optional long       │
│ ❌ │ 2: patient_id: required int        │ 2: patient_id: optional long       │
│ ❌ │ 3: amount: required double         │ 3: amount: optional double         │
│ ✅ │ 4: insurance_code: optional string │ 4: insurance_code: optional string │
│ ❌ │ 5: billing_date: required date     │ 5: billing_date: optional date     │
└────┴────────────────────────────────────┴────────────────────────────────────┘



## Step 3: Verify

In [8]:
for table_id in catalog.list_tables("healthcare"):
    t = catalog.load_table(table_id)
    has_labels = bool(t.labels) if hasattr(t, 'labels') else False
    n_cols = len(list(t.schema().fields))
    print(f"{table_id[1]:20s}  columns={n_cols}  labels={'yes' if has_labels else 'no'}")
    if has_labels:
        for k, v in t.table_labels.items():
            print(f"  {k}: {v}")

print("\nSetup complete! Open demo.ipynb or governance.ipynb.")

patients              columns=6  labels=no
visits_summary        columns=5  labels=no
billing               columns=5  labels=no

Setup complete! Open demo.ipynb or governance.ipynb.
